tutorial
run a hyperparameter tuning while training a model
log every hyperparameter and metrics in mlflow ui
compare the results of various runs in ui
choose the best and register mmodel 


In [1]:
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import fetch_california_housing
import numpy as np
housing = fetch_california_housing()
print(housing.DESCR)

.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib:
https://lib.stat.cmu.edu/datasets/houses.zip

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 U.S. census, using one row per census
block group. A block g

In [2]:
#prepare the dataset 
data =pd.DataFrame(housing.data, columns=housing.feature_names)
data.head

<bound method NDFrame.head of        MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0      8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1      8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2      7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3      5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4      3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   
...       ...       ...       ...        ...         ...       ...       ...   
20635  1.5603      25.0  5.045455   1.133333       845.0  2.560606     39.48   
20636  2.5568      18.0  6.114035   1.315789       356.0  3.122807     39.49   
20637  1.7000      17.0  5.205543   1.120092      1007.0  2.325635     39.43   
20638  1.8672      18.0  5.329513   1.171920       741.0  2.123209     39.43   
20639  2.3886      16.0  5.254717   1.162264      1387.0  2.616981     39.37   

       Lo

In [3]:
data['price'] = housing.target

In [4]:
data.head(5)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [5]:
from urllib.parse import urlparse #mlflow experiment we use 
data

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422
...,...,...,...,...,...,...,...,...,...
20635,1.5603,25.0,5.045455,1.133333,845.0,2.560606,39.48,-121.09,0.781
20636,2.5568,18.0,6.114035,1.315789,356.0,3.122807,39.49,-121.21,0.771
20637,1.7000,17.0,5.205543,1.120092,1007.0,2.325635,39.43,-121.22,0.923
20638,1.8672,18.0,5.329513,1.171920,741.0,2.123209,39.43,-121.32,0.847


In [6]:
X = data.drop('price', axis=1)
y = data['price']


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
from mlflow.models import infer_signature
signature = infer_signature(X_train, y_train)

In [8]:
def hyperparameter_tuning(X_train, y_train):
    # Define the hyperparameter grid
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 5, 10],
        'min_samples_split': [2, 5, 10]
    }

    # Create a Random Forest Regressor
    rf = RandomForestRegressor(random_state=42)

    # Create a GridSearchCV object
    grid_search = GridSearchCV(estimator=rf, param_grid=param_grid,
                               cv=5, n_jobs=-1, scoring='neg_mean_squared_error')

    # Fit the model to the training data
    grid_search.fit(X_train, y_train)

    # Return the best hyperparameters
    return grid_search.best_params_

In [9]:
with mlflow.start_run() as run:
    best_params = hyperparameter_tuning(X_train, y_train)
    mlflow.log_params(best_params)

    # Create a Random Forest Regressor with the best hyperparameters
    rf_best = RandomForestRegressor(**best_params, random_state=42)

    # Fit the model to the training data
    rf_best.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred = rf_best.predict(X_test)

    # Calculate the mean squared error
    mse = mean_squared_error(y_test, y_pred)
    mlflow.log_metric("mse", mse)

    # Log the model
    mlflow.sklearn.log_model(rf_best, "model", signature=signature)
    mlflow.log_parameters(best_params)
    mlflow.log_metric("mse", mse)
    mlflow.log_param("best_n_estimators", best_params['n_estimators'])
    mlflow.log_param("best_max_depth", best_params['max_depth'])
    mlflow.log_param("best_min_samples_split", best_params['min_samples_split'])
    mlflow.log_param("best_min_samples_leaf", best_params['min_samples_leaf'])

  # tracking uri
    mlflow.set_tracking_uri(uri ="http://127.0.0.1:5000")
    #down one will give file if we didnt set up one
    tracking_uri_type_store = urlparse(mlflow.get_tracking_uri()).scheme
    if tracking_uri_type_store != "file":
        mlflow.sklearn.log_model(rf_best, "model", registered_model_name="RandomForestRegressor")
    else:
        mlflow.sklearn.log_model(rf_best, "model", signature=signature)
    
    print(f"best hyperparameters: {best_params}")
    print(f"Mean Squared Error: {mse}")



2026/10/16 15:45:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/16 15:46:31 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


AttributeError: module 'mlflow' has no attribute 'log_parameters'